# Fine-Tuning Qwen3.5-0.8B (LoRA bf16 / QLoRA) — Chatbot Medis ID (Pivot 6, jurnal)

Studi **scaling parameter satu keluarga** — Qwen3.5 **0.8B / 2B / 4B** (arsitektur & tokenizer **identik** → sumbu parameter terkontrol) untuk QA medis Bahasa Indonesia.

Dataset **ID native-only terbersih** `Data/processed_id_final/` (20000/2947/2952, 0 noise/dup/leak). Toolchain **Unsloth + TRL + PEFT**.

**Presisi via `LOAD_IN_4BIT` (sel Konfigurasi):** `False` (default) = **LoRA bf16** utk **H100/A100 ≥40GB**; `True` = QLoRA 4-bit utk GPU 24GB. Effective batch dikunci 16 di kedua mode.

> Loader model ini: **FastLanguageModel** — Qwen3.5-0.8B = model **teks** → `FastLanguageModel`. `dtype=None` auto-precision.

> Harness IDENTIK antar model; beda hanya identitas + loader + turn-token (ChatML `<|im_start|>` + scaffold `<think>`).

**Cara pakai:** jalankan sel berurutan. Mulai `RUN_MODE="pilot"` → baca **GATE** → kalau PASS set `RUN_MODE="full"`, Restart & run. Detail di `README_TRAIN.md`.

## 1. Install (Unsloth/TRL/PEFT/bnb) — lalu **Runtime > Restart**

Tanpa `%%capture` agar konflik versi terlihat.

In [ ]:
# Install Unsloth + TRL + PEFT + bnb. TANPA %%capture supaya error/konflik install terlihat.
# RunPod/Colab sudah punya torch CUDA -> --no-deps utk paket inti agar torch tak ke-replace.
# Map xformers per torch. Dep baru Unsloth (tyro/msgspec/cut_cross_entropy/torchao) dipasang eksplisit.
import os, re, torch
v = re.match(r'\d+\.\d+', str(torch.__version__)).group(0)
xf_map = {'2.11':'0.0.35','2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2','2.6':'0.0.29.post3','2.5':'0.0.29.post3','2.4':'0.0.27.post2'}
xformers = 'xformers==' + xf_map.get(v, '0.0.35')
if v not in xf_map:
    print(f"[WARN] torch {v} tak ada di map -> default {xformers} (cek bila import gagal).")
print(f"torch={torch.__version__} -> {xformers}")
!pip install -q sentencepiece protobuf "huggingface_hub>=0.34.0" hf_transfer langdetect rouge_score
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft triton unsloth tyro msgspec cut_cross_entropy torchao
!pip install -q transformers==4.56.2
!pip install -q --no-deps trl==0.22.2
!pip install -q "datasets>=3.4.1,<4.0.0"
import torch; torch._dynamo.config.recompile_limit = 64
print("install selesai — Runtime > Restart, lalu jalankan sel Verifikasi.")

## 1b. Verifikasi import (setelah Restart, sebelum Konfigurasi)

In [ ]:
# === Verifikasi (jalankan SETELAH Runtime > Restart, sebelum Konfigurasi) ===
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
import unsloth, transformers, trl, datasets, xformers
from unsloth import FastLanguageModel
for m in (unsloth, transformers, trl, datasets, xformers):
    print(m.__name__, m.__version__)
print("OK semua import sukses. Loader model ini:", "FastLanguageModel")

## 2. Environment (single GPU, seed)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single GPU — hindari multi-GPU surprise
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
import torch, random, numpy as np, json, glob, math
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 3. Konfigurasi — set `RUN_MODE` di sini

`pilot` = subset 1500/200, ~250 step. `full` = data penuh + EarlyStopping. Turn-token model ini: ChatML `<|im_start|>` + scaffold `<think>`.

In [ ]:
# ====================== IDENTITAS MODEL ======================
MODEL_ID          = "unsloth/Qwen3.5-0.8B"            # model TEKS ~0.8B -> FastLanguageModel
MODEL_ID_FALLBACK = "unsloth/Qwen3.5-0.8B-Instruct"   # cadangan bila id utama gagal

# ====================== RUN MODE ======================
# "pilot" = cek sinyal cepat (subset, ~250 step). "full" = training penuh.
# MULAI dari "pilot", baca GATE, baru ganti ke "full".
RUN_MODE = "pilot"      # "pilot" | "full"

# ====================== HYPERPARAMETER ======================
MAX_SEQ_LENGTH = 1024            # p99 antar-model ~931 -> aman

# ---- PRESISI (saklar utama H100 vs GPU kecil) ----
# False (DEFAULT) = LoRA bf16 murni  -> H100/A100, VRAM >=40GB (base 16-bit, tanpa artefak kuantisasi)
# True            = QLoRA 4-bit bnb  -> GPU 24GB (RTX4090/L4/A5000)
LOAD_IN_4BIT   = False

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGET = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

LEARNING_RATE = 1e-4             # dikunci (2e-4 terbukti terlalu agresif di studi ini)
EPOCHS        = 3                # full-mode; + EarlyStopping(patience=3)
WARMUP_RATIO  = 0.10
WEIGHT_DECAY  = 0.01
# Effective batch DIKUNCI = 16 di kedua mode (komparabilitas hyperparameter antar-run/antar-model).
if LOAD_IN_4BIT:                 # GPU 24GB: batch kecil + akumulasi. Model 4B: 1x16 bila OOM.
    PER_DEVICE_BATCH, GRAD_ACCUM = 2, 8
else:                            # H100 80GB bf16: batch besar, akumulasi kecil (16x1 juga boleh).
    PER_DEVICE_BATCH, GRAD_ACCUM = 8, 2
MAX_GRAD_NORM    = 1.0           # grad clipping (anti loss meledak)

# ====================== FORMAT TURN (Qwen3.5) ======================
# Template Qwen sisip <think>\n\n</think> kosong walau enable_thinking=False -> RESPONSE_PART memuat scaffold itu agar loss hanya di jawaban (train==infer).
INSTRUCTION_PART       = "<|im_start|>user\n"   # penanda giliran user
RESPONSE_PART          = "<|im_start|>assistant\n<think>\n\n</think>\n\n"      # penanda giliran asisten (loss mulai di sini)
EXPECTED_PROMPT_TAIL   = "</think>"      # ekor prompt inferensi (utk SELF-CHECK decode)
MERGE_SYSTEM_INTO_USER = False         # True hanya utk model tanpa role system (Qwen: False)

# ====================== PILOT SUBSET ======================
PILOT_TRAIN_N, PILOT_VAL_N, PILOT_MAX_STEPS = 1500, 200, 250

# ====================== OUTPUT ======================
ADAPTER_DIR = "checkpoints/qwen35_0.8b_p6"
print(f"RUN_MODE={RUN_MODE} | model={MODEL_ID} | eff_batch={PER_DEVICE_BATCH*GRAD_ACCUM} | seq={MAX_SEQ_LENGTH}")

## 4. Lokasi dataset beku (RunPod/Kaggle/Colab/lokal)

Upload `train/val/test.jsonl` (**processed_id_final**) ke volume RunPod `/workspace` atau Drive — lihat `README_TRAIN.md`. Sel ini mencari otomatis (atau set env `DATA_DIR`).

In [ ]:
# Cari dataset beku (RunPod volume / Kaggle input / Colab Drive / lokal). Lihat README_TRAIN.md.
def _first_existing(paths):
    for p in paths:
        if p and os.path.exists(os.path.join(p, "train.jsonl")):
            return p
    return None

CANDIDATES = [
    os.environ.get("DATA_DIR", ""),
    "/workspace/processed_id_final",               # RunPod: unggah ke volume /workspace
    "/workspace/Data/processed_id_final",
    "/kaggle/input/processed-id-final",            # Kaggle dataset (rename sesuai uploadmu)
    "/content/drive/MyDrive/Fine-Tune SLM for Medical Chatbot/Data/processed_id_final",
    "../Data/processed_id_final", "Data/processed_id_final",
]
# Colab: mount Drive bila perlu
if any("drive/MyDrive" in c for c in CANDIDATES) and not os.path.exists("/content/drive"):
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception:
        pass
DATA_DIR = _first_existing(CANDIDATES)
assert DATA_DIR, ("Dataset tak ketemu. Set env DATA_DIR atau upload "
                  "train/val/test.jsonl (processed_id_final). Lihat README_TRAIN.md.")
os.makedirs(ADAPTER_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)

## 5. Load model (FastLanguageModel) — presisi mengikuti `LOAD_IN_4BIT`

Qwen3.5-0.8B = model **teks** → `FastLanguageModel`. `dtype=None` auto-precision.

> `TOK`/`PAD_ID` disiapkan agar encode/decode aman baik untuk tokenizer murni (Qwen3.5-0.8B) maupun processor VLM (Qwen3.5-2B/4B).

In [ ]:
from unsloth import FastLanguageModel

def _load(mid):
    # Checkpoint "-bnb-4bit" adalah pre-quantized -> WAJIB load_in_4bit=True walau mode bf16.
    four = LOAD_IN_4BIT or ("bnb-4bit" in mid)
    if four and not LOAD_IN_4BIT:
        print(f"[WARN] {mid} = checkpoint pre-quantized -> dipaksa QLoRA 4-bit (bukan bf16 murni). "
              "Hasil run ini JANGAN dicampur dgn run bf16 di tabel jurnal.")
    return FastLanguageModel.from_pretrained(
        model_name     = mid,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = None,          # auto: bf16 di Ampere/Hopper (H100), fp16 di GPU lama
        load_in_4bit   = four,
    )

try:
    model, tokenizer = _load(MODEL_ID); print("Loaded:", MODEL_ID)
except Exception as e:
    print("Gagal:", repr(e)[:160], "-> fallback", MODEL_ID_FALLBACK)
    model, tokenizer = _load(MODEL_ID_FALLBACK)

_q = bool(getattr(model, "is_loaded_in_4bit", False))
print("precision:", "QLoRA 4-bit (bnb)" if _q else f"LoRA {next(model.parameters()).dtype}",
      "| target mode:", "4-bit" if LOAD_IN_4BIT else "bf16")
assert _q == LOAD_IN_4BIT or not LOAD_IN_4BIT, "Mode 4-bit diminta tapi model tidak ter-quantize?"

# VLM (Qwen3.5-2B/4B via FastModel) bisa mengembalikan PROCESSOR, bukan tokenizer murni.
# TOK = tokenizer teks di baliknya utk encode/decode/pad; templating tetap via `tokenizer`
# (processor mendelegasikan apply_chat_template). Utk Qwen0.8B (teks), TOK identik dgn tokenizer.
TOK = getattr(tokenizer, "tokenizer", tokenizer)
if getattr(TOK, "pad_token", None) is None:
    TOK.pad_token = TOK.eos_token
TOK.padding_side = "right"
try:
    tokenizer.padding_side = "right"
except Exception:
    pass
PAD_ID = TOK.pad_token_id
print("loader: FastLanguageModel | vocab:", len(TOK), "| pad:", TOK.pad_token,
      "| processor?", TOK is not tokenizer)

## 6. LoRA adapter

**Catat `print_trainable_parameters()`** (dipakai di tabel jurnal — jumlah parameter aktif per model).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = LORA_TARGET,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
)
model.print_trainable_parameters()   # CATAT angka ini utk tabel jurnal (RQ parameter)

## 7. Dataset → teks (chat template Qwen3.5)

Qwen mendukung role `system` native (tidak di-merge).

In [ ]:
from datasets import load_dataset

def to_model_messages(messages):
    # Utk model tanpa role 'system' -> merge ke giliran user pertama.
    # Qwen3.5 (semua ukuran): system native -> no-op (MERGE_SYSTEM_INTO_USER=False).
    if not MERGE_SYSTEM_INTO_USER:
        return messages
    sys_txt, out, injected = "", [], False
    for m in messages:
        if m["role"] == "system":
            sys_txt = m["content"]; continue
        if m["role"] == "user" and not injected and sys_txt:
            out.append({"role": "user", "content": sys_txt.strip() + "\n\n" + m["content"]})
            injected = True
        else:
            out.append(m)
    if sys_txt and not injected:
        out.insert(0, {"role": "user", "content": sys_txt})
    return out

def render_chat(messages, add_generation_prompt=False):
    msgs = to_model_messages(messages)
    # Qwen: enable_thinking=False (template tetap sisip <think></think> kosong, di-mask via RESPONSE_PART).
    # Bila template tak terima kwarg ini -> TypeError -> fallback tanpa kwarg.
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                   add_generation_prompt=add_generation_prompt, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                   add_generation_prompt=add_generation_prompt)

def formatting_func(ex):
    return {"text": render_chat(ex["messages"], add_generation_prompt=False)}

train_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "train.jsonl"), split="train")
val_ds   = load_dataset("json", data_files=os.path.join(DATA_DIR, "val.jsonl"),   split="train")

if RUN_MODE == "pilot":
    train_ds = train_ds.shuffle(seed=SEED).select(range(min(PILOT_TRAIN_N, len(train_ds))))
    val_ds   = val_ds.shuffle(seed=SEED).select(range(min(PILOT_VAL_N, len(val_ds))))

train_ds = train_ds.map(formatting_func, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(formatting_func,   remove_columns=val_ds.column_names)
print(f"RUN_MODE={RUN_MODE} | train={len(train_ds)} | val={len(val_ds)}")

## 8. SELF-CHECK decode (WAJIB dilihat sebelum train)

Membuktikan token **konten pertama jawaban** ikut dipelajari (bukan ke-mask/off-by-one) dan format turn membuat **train == inferensi**.

In [ ]:
# ===== SELF-CHECK decode (sama logika dgn validate_dataset_for_training.py) =====
# Pastikan: token KONTEN pertama jawaban IKUT dipelajari (tak ke-mask / off-by-one),
# format turn konsisten train==inferensi. Encode/decode via TOK (aman utk processor VLM).
def _find_sub(seq, sub):
    for i in range(len(seq)-len(sub)+1):
        if seq[i:i+len(sub)] == sub: return i
    return -1

_rec   = json.loads(open(os.path.join(DATA_DIR, "train.jsonl"), encoding="utf-8").readline())
_msgs  = _rec["messages"]
_full  = render_chat(_msgs, add_generation_prompt=False)
_prompt= render_chat([m for m in _msgs if m["role"]!="assistant"], add_generation_prompt=True)
_ids   = TOK(_full, add_special_tokens=False)["input_ids"]
_marker= TOK(RESPONSE_PART, add_special_tokens=False)["input_ids"]
_pos   = _find_sub(_ids, _marker)
assert _pos >= 0, "RESPONSE_PART tak ditemukan di teks ter-render! Cek template/turn-token model."
_start = _pos + len(_marker)
_ans   = next(m["content"] for m in _msgs if m["role"]=="assistant")
_w1    = _ans.split()[0] if _ans.split() else ""
_reco  = TOK.decode(_ids[_start:_start+12]).lstrip()
print("first ACTIVE tokens :", [TOK.decode([t]) for t in _ids[_start:_start+8]])
print("jawaban kata-1 asli :", repr(_w1), "| rekonstruksi:", repr(_reco[:40]))
print("infer prompt ends   :", repr(_prompt.rstrip()[-40:]))
assert _reco.lower().startswith(_w1[:6].lower()), "OFF-BY-ONE: token jawaban pertama ke-mask/terpotong!"
assert _prompt.rstrip().endswith(EXPECTED_PROMPT_TAIL), \
    f"Inferensi TIDAK berakhir {EXPECTED_PROMPT_TAIL!r} -> train != infer!"
print("OK decode-check: jawaban medis pertama dipelajari, train==inferensi konsisten.")

## 9. Konfigurasi training (SFT + train_on_responses_only)

`train_on_responses_only` → loss **hanya pada jawaban**. `packing=False` (wajib). RunPod Ampere+ → `bf16=True`. Unsloth otomatis patch cross-entropy.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only

USE_BF16 = torch.cuda.is_bf16_supported()   # RunPod Ampere+ -> True

# cadence & durasi tergantung RUN_MODE
if RUN_MODE == "pilot":
    steps_kw = dict(max_steps=PILOT_MAX_STEPS, num_train_epochs=1,
                    eval_steps=50, save_steps=50)
else:
    steps_per_epoch = max(1, math.ceil(len(train_ds) / (PER_DEVICE_BATCH*GRAD_ACCUM)))
    es = max(10, steps_per_epoch // 10)     # ~10x/epoch
    steps_kw = dict(num_train_epochs=EPOCHS, eval_steps=es, save_steps=es)

cfg = SFTConfig(
    output_dir                  = ADAPTER_DIR,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    per_device_eval_batch_size  = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_ratio                = WARMUP_RATIO,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    weight_decay                = WEIGHT_DECAY,
    optim                       = "adamw_8bit",
    fp16                        = not USE_BF16,
    bf16                        = USE_BF16,
    max_grad_norm               = MAX_GRAD_NORM,   # grad clipping
    logging_steps               = 10,
    eval_strategy               = "steps",
    save_strategy               = "steps",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    seed                        = SEED, data_seed = SEED,
    dataset_text_field          = "text",
    max_length                  = MAX_SEQ_LENGTH,
    packing                     = False,           # WAJIB utk train_on_responses_only
    report_to                   = "none",
    **steps_kw,
)

_cb = [] if RUN_MODE == "pilot" else [EarlyStoppingCallback(3, 0.001)]
_kw = dict(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=cfg, callbacks=_cb)
try:
    trainer = SFTTrainer(processing_class=tokenizer, **_kw)
except TypeError:
    trainer = SFTTrainer(tokenizer=tokenizer, **_kw)

# loss HANYA pada jawaban medis (penanda turn diverifikasi di sel SELF-CHECK)
trainer = train_on_responses_only(trainer,
    instruction_part=INSTRUCTION_PART, response_part=RESPONSE_PART)

# Unsloth otomatis mem-patch cross-entropy (fused) -> hemat VRAM, tanpa compute_loss_func manual.
print("trainer siap. RUN_MODE =", RUN_MODE, "| steps_kw =", steps_kw)

## 10. Train (resume-aware)

In [ ]:
# Resume bila checkpoint ada (tahan disconnect RunPod/Colab).
_ckpts = glob.glob(os.path.join(ADAPTER_DIR, "checkpoint-*"))
_resume = len(_ckpts) > 0
print("resume:", _resume, f"({len(_ckpts)} checkpoint)")
trainer_stats = trainer.train(resume_from_checkpoint=_resume or None)
print("training selesai.")

## 11. GATE (setelah PILOT) — PASS/STOP

Cek: loss turun & tak meledak/NaN; 10 generasi ≥9/10 Indonesia (anti language-mixing), tidak degeneratif. STOP → `README_TRAIN.md` › TROUBLESHOOTING.

In [ ]:
# ===== GATE (jalankan setelah PILOT). Cetak PASS / STOP. =====
from langdetect import detect, DetectorFactory; DetectorFactory.seed = 42

logs = trainer.state.log_history
losses = [l["loss"] for l in logs if "loss" in l]
def _mean(x): return sum(x)/len(x) if x else float("nan")
k = max(1, len(losses)//5)
loss_start, loss_end = _mean(losses[:k]), _mean(losses[-k:])
loss_max = max(losses) if losses else float("nan")
B2_trend = (loss_end <= loss_start - 0.15)
B2_noexplode = (loss_max <= 2*losses[0]) if losses else False
B1_stable = all(np.isfinite(l) for l in losses) and len(losses) > 0
print(f"[B1] stabil(no NaN/Inf): {B1_stable}")
print(f"[B2] loss {loss_start:.3f} -> {loss_end:.3f} (turun>=0.15: {B2_trend}; tak meledak: {B2_noexplode})")

# B3 generasi: 10 prompt tetap dari test
FastLanguageModel.for_inference(model)
test_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "test.jsonl"), split="train")
gens = []
id_ok = degen = 0
for i in range(10):
    msgs = [m for m in test_ds[i]["messages"] if m["role"] != "assistant"]
    p = render_chat(msgs, add_generation_prompt=True)
    inp = TOK(p, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=200, do_sample=False, no_repeat_ngram_size=3,
                         repetition_penalty=1.1, pad_token_id=PAD_ID)
    txt = TOK.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gens.append((msgs[-1]["content"], txt))
    try:
        if detect(txt[:500]) == "id": id_ok += 1
    except Exception:
        pass
    toks = txt.split()
    grams = [tuple(toks[j:j+4]) for j in range(len(toks)-3)]
    rep = (1 - len(set(grams))/len(grams)) if grams else 0
    if rep > 0.30: degen += 1
B3a_lang = id_ok >= 9
B3b_degen = degen == 0
print(f"[B3a] Indonesia {id_ok}/10 (anti language-mixing: {B3a_lang})")
print(f"[B3b] non-degeneratif {10-degen}/10 (no >30% 4-gram berulang: {B3b_degen})")

with open("pilot_generations.txt", "w", encoding="utf-8") as f:
    for q, a in gens:
        f.write(f"Q: {q}\nA: {a}\n{'-'*60}\n")
print("contoh 3 generasi:")
for q, a in gens[:3]:
    print("Q:", q[:80]); print("A:", a[:160]); print("-"*40)

PASS = B1_stable and B2_trend and B2_noexplode and B3a_lang and B3b_degen
print("\n" + "="*60)
print("GATE:", "PASS_GREEN -> boleh set RUN_MODE='full'" if PASS
      else "STOP / NEEDS_HUMAN -> lihat TROUBLESHOOTING di README_TRAIN.md")
print("="*60)
FastLanguageModel.for_training(model)

## 12. Simpan adapter + generasi

In [ ]:
# Simpan adapter LoRA (kecil) utk inspeksi/arsip. (Merge 16-bit dilakukan saat eval.)
print("best checkpoint:", trainer.state.best_model_checkpoint)
print("best eval_loss :", trainer.state.best_metric)
model.save_pretrained(ADAPTER_DIR)
try:
    tokenizer.save_pretrained(ADAPTER_DIR)
except Exception:
    TOK.save_pretrained(ADAPTER_DIR)
print("adapter ->", ADAPTER_DIR, "| 10 generasi -> pilot_generations.txt")

## 13. (Placeholder) Eval token-F1 + ROUGE-L (val)

Eval final lintas-model via `eval.py` (token-F1 + ROUGE-L + BERTScore, per-bahasa). Sel ini hanya cek cepat pada val setelah training.

In [ ]:
# ===== (PLACEHOLDER) eval token-F1 + ROUGE-L pada val — siap pakai setelah training =====
# Metrik utama studi = token-F1 + ROUGE-L (+ BERTScore di eval.py). EM/MCQA di-DROP (native-only).
!pip install -q rouge-score
from rouge_score import rouge_scorer
import collections

def _f1(pred, ref):
    p, r = pred.split(), ref.split()
    common = collections.Counter(p) & collections.Counter(r)
    ns = sum(common.values())
    if ns == 0 or not p or not r: return 0.0
    prec, rec = ns/len(p), ns/len(r)
    return 2*prec*rec/(prec+rec)

FastLanguageModel.for_inference(model)
val_eval = load_dataset("json", data_files=os.path.join(DATA_DIR, "val.jsonl"), split="train")
N = min(100, len(val_eval))
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
f1s, rls = [], []
for i in range(N):
    msgs = val_eval[i]["messages"]
    ref = next(m["content"] for m in msgs if m["role"]=="assistant")
    p = render_chat([m for m in msgs if m["role"]!="assistant"], add_generation_prompt=True)
    inp = TOK(p, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=256, do_sample=False, no_repeat_ngram_size=3,
                         repetition_penalty=1.1, pad_token_id=PAD_ID)
    pred = TOK.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    f1s.append(_f1(pred, ref)); rls.append(scorer.score(ref, pred)["rougeL"].fmeasure)
print(f"[EVAL n={N}] token-F1={sum(f1s)/N:.4f} | ROUGE-L={sum(rls)/N:.4f}")
FastLanguageModel.for_training(model)

---
**Selesai.** Qwen3.5-0.8B — bagian studi scaling parameter Qwen3.5 (0.8B / 2B / 4B, arsitektur & tokenizer identik). Semua dilatih di `processed_id_final`, dievaluasi dengan protokol identik.